# QPU Sentinel — Experiment 001

## Quantum Simulator Validation

**Objective:** Verify that our Qiskit installation can construct,
transpile and execute a quantum circuit using Qiskit Aer.

We will create a two-qubit Bell state and measure its outcomes.

In [1]:
# Experiment 001: Bell State Simulator Test

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Create a two-qubit circuit
qc = QuantumCircuit(2)

# Create a Bell state
qc.h(0)
qc.cx(0, 1)

# Measure both qubits
qc.measure_all()

# Initialize simulator
simulator = AerSimulator()

# Compile and execute
compiled_circuit = transpile(qc, simulator)

job = simulator.run(
    compiled_circuit,
    shots=1024,
    seed_simulator=42
)

result = job.result()
counts = result.get_counts()

print("Measurement results:", counts)
print("Total shots:", sum(counts.values()))
print("Circuit depth:", compiled_circuit.depth())
print("Two-qubit gates:", compiled_circuit.num_nonlocal_gates())

assert sum(counts.values()) == 1024
assert set(counts.keys()).issubset({"00", "11"})

print("\nSUCCESS: Qiskit Aer simulation completed!")

Measurement results: {'00': 521, '11': 503}
Total shots: 1024
Circuit depth: 3
Two-qubit gates: 1

SUCCESS: Qiskit Aer simulation completed!


# Experiment 002 — Financial Portfolio Optimization

## Problem definition

We want to select exactly 2 assets from a portfolio of 4.

Our objective is to balance expected return against portfolio risk.

We will use hypothetical annualized financial data, not live market data.

### Optimization objective

Minimize:

λ xᵀΣx − μᵀx

Subject to:

Σ xᵢ = K

xᵢ ∈ {0,1}

Where:
- μ = expected returns
- Σ = covariance matrix
- λ = risk-aversion parameter
- K = number of assets to select
- xᵢ = 1 if an asset is selected, otherwise 0

In [2]:
import numpy as np
import pandas as pd

# Four hypothetical financial assets
assets = ["ASSET_A", "ASSET_B", "ASSET_C", "ASSET_D"]

# Hypothetical annual expected returns
expected_returns = np.array([
    0.12,
    0.10,
    0.07,
    0.15
])

# Hypothetical annual covariance matrix
covariance = np.array([
    [0.0400, 0.0060, 0.0040, 0.0080],
    [0.0060, 0.0225, 0.0030, 0.0050],
    [0.0040, 0.0030, 0.0100, 0.0020],
    [0.0080, 0.0050, 0.0020, 0.0625]
])

# Portfolio constraints
K = 2
risk_aversion = 2.0

# Display data
portfolio_data = pd.DataFrame({
    "Asset": assets,
    "Expected Annual Return": expected_returns
})

print(portfolio_data)
print("\nCovariance Matrix:")
print(pd.DataFrame(covariance, index=assets, columns=assets))

print("\nCovariance matrix eigenvalues:")
print(np.linalg.eigvalsh(covariance))

assert np.all(np.linalg.eigvalsh(covariance) >= -1e-10)

print("\nFinancial input validation passed!")

     Asset  Expected Annual Return
0  ASSET_A                    0.12
1  ASSET_B                    0.10
2  ASSET_C                    0.07
3  ASSET_D                    0.15

Covariance Matrix:
         ASSET_A  ASSET_B  ASSET_C  ASSET_D
ASSET_A    0.040   0.0060    0.004   0.0080
ASSET_B    0.006   0.0225    0.003   0.0050
ASSET_C    0.004   0.0030    0.010   0.0020
ASSET_D    0.008   0.0050    0.002   0.0625

Covariance matrix eigenvalues:
[0.00907687 0.02072788 0.03889644 0.06629881]

Financial input validation passed!


In [3]:
from itertools import product

def portfolio_objective(x):
    risk = x @ covariance @ x
    returns = expected_returns @ x

    return risk_aversion * risk - returns

classical_results = []

for bits in product([0, 1], repeat=len(assets)):
    x = np.array(bits)

    if np.sum(x) == K:
        risk = x @ covariance @ x
        returns = expected_returns @ x
        objective = risk_aversion * risk - returns

        classical_results.append({
            "Selection": "".join(map(str, bits)),
            "Assets": [
                assets[i] for i in range(len(assets)) if x[i] == 1
            ],
            "Expected Return": returns,
            "Portfolio Variance": risk,
            "Objective": objective
        })

classical_results = sorted(
    classical_results,
    key=lambda row: row["Objective"]
)

benchmark = pd.DataFrame(classical_results)

print("CLASSICAL EXACT OPTIMIZATION RESULTS")
display(benchmark)

print("\nBest portfolio:")
print(benchmark.iloc[0])

CLASSICAL EXACT OPTIMIZATION RESULTS


,Selection,Assets,Expected Return,Portfolio Variance,Objective
0,0110,"[ASSET_B, ASSET_C]",0.17,0.0385,-0.093
1,1010,"[ASSET_A, ASSET_C]",0.19,0.0580,-0.074
2,1100,"[ASSET_A, ASSET_B]",0.22,0.0745,-0.071
3,0011,"[ASSET_C, ASSET_D]",0.22,0.0765,-0.067
4,0101,"[ASSET_B, ASSET_D]",0.25,0.0950,-0.060
5,1001,"[ASSET_A, ASSET_D]",0.27,0.1185,-0.033



Best portfolio:
Selection                           0110
Assets                [ASSET_B, ASSET_C]
Expected Return                     0.17
Portfolio Variance                0.0385
Objective                         -0.093
Name: 0, dtype: object


In [4]:
from itertools import product

def portfolio_objective(x):
    risk = x @ covariance @ x
    returns = expected_returns @ x

    return risk_aversion * risk - returns

classical_results = []

for bits in product([0, 1], repeat=len(assets)):
    x = np.array(bits)

    if np.sum(x) == K:
        risk = x @ covariance @ x
        returns = expected_returns @ x
        objective = risk_aversion * risk - returns

        classical_results.append({
            "Selection": "".join(map(str, bits)),
            "Assets": [
                assets[i] for i in range(len(assets)) if x[i] == 1
            ],
            "Expected Return": returns,
            "Portfolio Variance": risk,
            "Objective": objective
        })

classical_results = sorted(
    classical_results,
    key=lambda row: row["Objective"]
)

benchmark = pd.DataFrame(classical_results)

print("CLASSICAL EXACT OPTIMIZATION RESULTS")
display(benchmark)

print("\nBest portfolio:")
print(benchmark.iloc[0])

CLASSICAL EXACT OPTIMIZATION RESULTS


,Selection,Assets,Expected Return,Portfolio Variance,Objective
0,0110,"[ASSET_B, ASSET_C]",0.17,0.0385,-0.093
1,1010,"[ASSET_A, ASSET_C]",0.19,0.0580,-0.074
2,1100,"[ASSET_A, ASSET_B]",0.22,0.0745,-0.071
3,0011,"[ASSET_C, ASSET_D]",0.22,0.0765,-0.067
4,0101,"[ASSET_B, ASSET_D]",0.25,0.0950,-0.060
5,1001,"[ASSET_A, ASSET_D]",0.27,0.1185,-0.033



Best portfolio:
Selection                           0110
Assets                [ASSET_B, ASSET_C]
Expected Return                     0.17
Portfolio Variance                0.0385
Objective                         -0.093
Name: 0, dtype: object


# Experiment 003 — QUBO Formulation

We transform the financial portfolio objective into a
Quadratic Unconstrained Binary Optimization problem.

The cardinality penalty encourages exactly two selected assets.

QUBO objective:

f(x) = λxᵀΣx - μᵀx + A(Σxᵢ - K)²

Where xᵢ ∈ {0,1}.

In [5]:
# Experiment 003: Build the QUBO

penalty = 1.0
n = len(assets)

# QUBO coefficients:
# E(x) = constant + sum(linear[i] * x[i])
#                  + sum(quadratic[i,j] * x[i] * x[j])

linear = np.zeros(n)
quadratic = np.zeros((n, n))

# Financial objective diagonal terms
for i in range(n):
    linear[i] = (
        risk_aversion * covariance[i, i]
        - expected_returns[i]
        + penalty * (1 - 2 * K)
    )

# Pairwise interactions
for i in range(n):
    for j in range(i + 1, n):
        quadratic[i, j] = (
            2 * risk_aversion * covariance[i, j]
            + 2 * penalty
        )

constant = penalty * K**2

def qubo_energy(x):
    energy = constant + np.dot(linear, x)

    for i in range(n):
        for j in range(i + 1, n):
            energy += quadratic[i, j] * x[i] * x[j]

    return energy

# Verify QUBO against original constrained objective
for bits in product([0, 1], repeat=n):
    x = np.array(bits)
    original = portfolio_objective(x) + penalty * (np.sum(x) - K)**2
    converted = qubo_energy(x)

    assert np.isclose(original, converted)

print("QUBO linear coefficients:", linear)
print("\nQUBO quadratic coefficients:")
print(quadratic)
print("\nConstant:", constant)
print("\nAll 16 QUBO configurations verified!")

QUBO linear coefficients: [-3.04  -3.055 -3.05  -3.025]

QUBO quadratic coefficients:
[[0.    2.024 2.016 2.032]
 [0.    0.    2.012 2.02 ]
 [0.    0.    0.    2.008]
 [0.    0.    0.    0.   ]]

Constant: 4.0

All 16 QUBO configurations verified!


In [6]:
qubo_results = []

for bits in product([0, 1], repeat=n):
    x = np.array(bits)

    qubo_results.append({
        "Selection": "".join(map(str, bits)),
        "Number Selected": int(np.sum(x)),
        "QUBO Energy": qubo_energy(x),
        "Feasible": np.sum(x) == K
    })

qubo_results = pd.DataFrame(qubo_results)
qubo_results = qubo_results.sort_values("QUBO Energy")

display(qubo_results)

print("\nBest QUBO configuration:")
display(qubo_results.iloc[0])

,Selection,Number Selected,QUBO Energy,Feasible
6,0110,2,-0.093,True
10,1010,2,-0.074,True
12,1100,2,-0.071,True
3,0011,2,-0.067,True
5,0101,2,-0.060,True
9,1001,2,-0.033,True
14,1110,3,0.907,False
7,0111,3,0.910,False
11,1011,3,0.941,False
4,0100,1,0.945,False



Best QUBO configuration:


Selection           0110
Number Selected        2
QUBO Energy       -0.093
Feasible            True
Name: 6, dtype: object

# Experiment 004 — QAOA portfolio optimization

This runs a depth-2 QAOA circuit using Qiskit and Aer. A standard X mixer is used with the cardinality penalty already included in the QUBO. SciPy COBYLA minimizes the sampled QUBO energy. The results below compare sampled feasible portfolios with the exact classical optimum and report simulator circuit metrics; this is not evidence of quantum advantage.

In [7]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.qaoa_portfolio import run_qaoa_portfolio

exact = benchmark.iloc[0]
qaoa_results, qaoa_metrics = run_qaoa_portfolio(
    linear=linear,
    quadratic=quadratic,
    cardinality=K,
    portfolio_objective=portfolio_objective,
    qubo_energy=qubo_energy,
    exact_selection=exact["Selection"],
    exact_objective=exact["Objective"],
)

for metric, value in qaoa_metrics.items():
    print(f"{metric}: {value}")
display(qaoa_results)

from src.hardware_suitability import CircuitRequirements, analyze_backends

required_couplings = tuple(
    (i, j)
    for i in range(len(assets))
    for j in range(i + 1, len(assets))
    if quadratic[i, j] != 0
)
hardware_suitability = analyze_backends(
    CircuitRequirements(
        qubits=len(assets),
        circuit_depth=qaoa_metrics["circuit_depth"],
        two_qubit_gates=qaoa_metrics["two_qubit_gates"],
        required_couplings=required_couplings,
    )
)
print("\nSIMULATED PROFILE SUITABILITY (not hardware measurements)")
print(hardware_suitability["recommendation"])
display(pd.DataFrame([
    {
        "Rank": rank,
        "Profile": profile["name"],
        "Type": profile["profile_type"],
        "Compatible": profile["compatible"],
        "Score": profile["score"],
        "Reasons": "; ".join(profile["reasons"]),
    }
    for rank, profile in enumerate(hardware_suitability["ranked_profiles"], 1)
]))

depth_p: 2
optimizer_success: True
optimizer_evaluations: 39
sampled_energy_estimate: 0.07979199218750001
circuit_depth: 15
two_qubit_gates: 12
final_shots: 8192
feasible_shots: 7068
feasibility_rate: 0.86279296875
exact_selection: 0110
exact_objective: -0.09300000000000001
best_feasible_selection: 0110
best_feasible_objective: -0.09300000000000001
optimal_shots: 1189
optimal_sample_rate: 0.1451416015625


,Selection,Shots,Probability,Feasible,Objective,QUBO Energy
0,0101,1235,0.150757,True,-0.060,-0.060
1,1001,1204,0.146973,True,-0.033,-0.033
2,0110,1189,0.145142,True,-0.093,-0.093
3,1010,1170,0.142822,True,-0.074,-0.074
4,1100,1162,0.141846,True,-0.071,-0.071
5,0011,1108,0.135254,True,-0.067,-0.067
6,1000,180,0.021973,False,-0.040,0.960
7,0001,160,0.019531,False,-0.025,0.975
8,0010,147,0.017944,False,-0.050,0.950
9,0100,135,0.016479,False,-0.055,0.945



SIMULATED PROFILE SUITABILITY (not hardware measurements)
Simulated Profile A - low-error line is the highest-scoring compatible simulated profile (82.17/100).


,Rank,Profile,Type,Compatible,Score,Reasons
0,1,Simulated Profile A - low-error line,SIMULATED PROFILE,True,82.17,Capacity: 5 profile qubits cover 4 circuit qub...
1,2,Simulated Profile B - balanced ring,SIMULATED PROFILE,True,79.80,Capacity: 7 profile qubits cover 4 circuit qub...
2,3,Simulated Profile C - compact dense,SIMULATED PROFILE,True,67.00,Capacity: 4 profile qubits cover 4 circuit qub...
